### Building A RAG system with langchain and Chromadb

In [27]:
import os

In [26]:
#langchain import 
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import Chroma

import numpy as np
from typing import List


#### Creating Sample Docs

In [28]:
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine Learning (ML) is a subset of Artificial Intelligence that enables
    computers to learn patterns from data without being explicitly programmed.
    It is widely used in applications such as recommendation systems, spam
    detection, fraud detection, predictive analytics, and image classification.
    Machine Learning algorithms can be supervised, unsupervised, or reinforcement
    learning based depending on the type of training data available.
    """,

    """
    Deep Learning Fundamentals
    
    Deep Learning (DL) is a specialized branch of Machine Learning that uses
    artificial neural networks with multiple hidden layers to learn complex
    patterns from large amounts of data. Deep Learning has achieved remarkable
    success in computer vision, speech recognition, autonomous vehicles, and
    medical image analysis. Popular frameworks for Deep Learning include
    TensorFlow and PyTorch.
    """,

    """
    NLP Fundamentals 
         
    Natural Language Processing (NLP) is a field of Artificial Intelligence
    that focuses on enabling computers to understand, interpret, and generate
    human language. NLP powers applications such as chatbots, language
    translation, sentiment analysis, text summarization, question answering,
    and virtual assistants like ChatGPT. Modern NLP systems often use
    Transformer-based models such as BERT and GPT.
    """
]

print(sample_docs[0])


    Machine Learning Fundamentals

    Machine Learning (ML) is a subset of Artificial Intelligence that enables
    computers to learn patterns from data without being explicitly programmed.
    It is widely used in applications such as recommendation systems, spam
    detection, fraud detection, predictive analytics, and image classification.
    Machine Learning algorithms can be supervised, unsupervised, or reinforcement
    learning based depending on the type of training data available.
    


In [29]:
### save sample doc to files
import tempfile
temp_dir = tempfile.mkdtemp()
for i,doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt","w") as f:
        f.write(doc)



print(f"sample Docs Created in {temp_dir}")        
        

sample Docs Created in C:\Users\mrraj\AppData\Local\Temp\tmplmzhwo7s


### Document Loading


In [31]:
from langchain_community.document_loaders import DirectoryLoader

loader = DirectoryLoader(
    temp_dir,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'}
)

documents = loader.load()
print(f"Loaded {len(documents)} documents")
print(f"\n First Document Preview:")
print(documents[0].page_content[:200] + "...")

Loaded 3 documents

 First Document Preview:

    Machine Learning Fundamentals

    Machine Learning (ML) is a subset of Artificial Intelligence that enables
    computers to learn patterns from data without being explicitly programmed.
    It ...


### Document SPlitting


In [32]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len,
    separators=[" "] 
)

chunks =splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f" Content :{chunks[0].page_content[:150]}")
print(f"Metadat:{chunks[0].metadata}")




Created 4 chunks from 3 documents
 Content :Machine Learning Fundamentals

    Machine Learning (ML) is a subset of Artificial Intelligence that enables
    computers to learn patterns from data
Metadat:{'source': 'C:\\Users\\mrraj\\AppData\\Local\\Temp\\tmplmzhwo7s\\doc_0.txt'}


### Embedding MOdel

In [33]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
  model_name = "sentence-transformers/all-miniLM-L6-V2"
)
embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='sentence-transformers/all-miniLM-L6-V2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [34]:
texts = [doc.page_content for doc in documents]

vectors = embeddings.embed_documents(texts)

print(f"Total Vectors : {len(vectors)}")
print(f"Dimension     : {len(vectors[0])}")
print(texts[0])
print(vectors)


Total Vectors : 3
Dimension     : 384

    Machine Learning Fundamentals

    Machine Learning (ML) is a subset of Artificial Intelligence that enables
    computers to learn patterns from data without being explicitly programmed.
    It is widely used in applications such as recommendation systems, spam
    detection, fraud detection, predictive analytics, and image classification.
    Machine Learning algorithms can be supervised, unsupervised, or reinforcement
    learning based depending on the type of training data available.
    
[[-0.07032269239425659, -0.04299352318048477, 0.01506850030273199, -0.004108191933482885, 0.0414230152964592, 0.012256313115358353, 0.028246209025382996, -0.07202592492103577, -0.039712462574243546, -0.02781774289906025, -0.03078901581466198, 0.02918364107608795, 0.05319993197917938, -0.03839479386806488, -0.04055100306868553, 0.03380488604307175, 0.0382370725274086, 0.021934321150183678, -0.04742281883955002, -0.07891533523797989, 0.022390510886907578, 

### Initialize the chromaDB vector and the store the chunks in vector representation  

In [35]:
presist_directory = "./chroma_db"
vectorestore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=presist_directory,
    collection_name="Rag_collection"
) 

### Testing Similarity Search

In [41]:
query = "What is Machine Learning?"

results = vectorestore.similarity_search(query, k=3)
results[0].page_content


'Machine Learning Fundamentals\n\n    Machine Learning (ML) is a subset of Artificial Intelligence that enables\n    computers to learn patterns from data without being explicitly programmed.\n    It is widely used in applications such as recommendation systems, spam\n    detection, fraud detection, predictive analytics, and image classification.\n    Machine Learning algorithms can be supervised, unsupervised, or reinforcement\n    learning based depending on the type of training data available.'

In [36]:
query = "What is NLP?"

results = vectorestore.similarity_search(query, k=3)
results[0].page_content


'NLP Fundamentals \n\n    Natural Language Processing (NLP) is a field of Artificial Intelligence\n    that focuses on enabling computers to understand, interpret, and generate\n    human language. NLP powers applications such as chatbots, language\n    translation, sentiment analysis, text summarization, question answering,\n    and virtual assistants like ChatGPT. Modern NLP systems often use\n    Transformer-based models such as BERT and GPT.'

### Advance similarity search with score

In [37]:
query = "What is Machine Learning?"
res = vectorestore.similarity_search_with_score(query,k=3)
for doc, score in res:
    print("=" * 60)
    print("Score :", score)
    print(doc.page_content)


Score : 0.46563035249710083
Machine Learning Fundamentals

    Machine Learning (ML) is a subset of Artificial Intelligence that enables
    computers to learn patterns from data without being explicitly programmed.
    It is widely used in applications such as recommendation systems, spam
    detection, fraud detection, predictive analytics, and image classification.
    Machine Learning algorithms can be supervised, unsupervised, or reinforcement
    learning based depending on the type of training data available.
Score : 0.46563035249710083
Machine Learning Fundamentals

    Machine Learning (ML) is a subset of Artificial Intelligence that enables
    computers to learn patterns from data without being explicitly programmed.
    It is widely used in applications such as recommendation systems, spam
    detection, fraud detection, predictive analytics, and image classification.
    Machine Learning algorithms can be supervised, unsupervised, or reinforcement
    learning based depend

### Inlitialize our LLM

In [1]:
from dotenv import load_dotenv
load_dotenv()
from langchain_google_genai import ChatGoogleGenerativeAI

In [9]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    temperature=0
)

response = llm.invoke("What is the capital of India?")

print(response.content)

[{'type': 'text', 'text': 'The capital of India is **New Delhi**.', 'extras': {'signature': 'EpoDCpcDARFNMg/4Asbk0kBOZOfLtuQQ5f0yZxgevtldE1OiH2rYvgPhDR3LtvOqSuok6vkS9HjiV7rxBZOYVoNsF5m5xFZhwRGbMBdNN49oKCkq/NT1LfDiZnshgHu4xVWtj0I72/LujwZZaxkmQedSyd3HoVL+fFPRPyYfEeYDjVcNieV6SkfBkqdhEMhPaGiJnWAigPrvJqRro8F6DUNgwWRt3h9geyqoU6zmSe3JtbB+0b8TBUcP6BK1MZflHJfpQNy0pFpV2WYOtkZPkkQOxAlrUsjc1hEgBN0oYUh/lawfhACCUF6y2gmIqUE6tpKMWE02hZ3j4czAmUItGUs1Cwar+5+vpYtunpL0XJdJZSkkaAwmhmmfBs8c3nf97ni6dFbTtxgup6s2UVmU7y9aFMCYG+pHYCPuaA6qQK26oosuQP7YOhN7GCVDtbpQJJNXl/nJtjmhJ1ZZ6JJfwyXKibCVwVYD4tpTAFPQzOxp9FL3Ekk8U+iDAsA6GGXQ1gV1QexpE3J6WuNb5VKfRFLtj2G4mvVv8h8joGc='}}]


### Modern Rag Chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
system_prompt = """
You are an expert AI assistant specialized in Retrieval-Augmented Generation (RAG).

Answer the user's question strictly based on the retrieved context provided below.

Instructions:
1. Use only the information available in the context.
2. Do not add assumptions or external knowledge.
3. If the answer cannot be found in the context, reply:
   "I don't have enough information in the provided context."
4. Keep your answers accurate, concise, and well-structured.
5. When appropriate, use bullet points or numbered lists for better readability.

Retrieved Context:
{context}
"""
prompt = ChatPromptTemplate.from_messages([
    ("system",system_prompt),
    ("human","{input}")
])
prompt
    


ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nYou are an expert AI assistant specialized in Retrieval-Augmented Generation (RAG).\n\nAnswer the user\'s question strictly based on the retrieved context provided below.\n\nInstructions:\n1. Use only the information available in the context.\n2. Do not add assumptions or external knowledge.\n3. If the answer cannot be found in the context, reply:\n   "I don\'t have enough information in the provided context."\n4. Keep your answers accurate, concise, and well-structured.\n5. When appropriate, use bullet points or numbered lists for better readability.\n\nRetrieved Context:\n{context}\n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_k

In [22]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain = create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\nYou are an expert AI assistant specialized in Retrieval-Augmented Generation (RAG).\n\nAnswer the user\'s question strictly based on the retrieved context provided below.\n\nInstructions:\n1. Use only the information available in the context.\n2. Do not add assumptions or external knowledge.\n3. If the answer cannot be found in the context, reply:\n   "I don\'t have enough information in the provided context."\n4. Keep your answers accurate, concise, and well-structured.\n5. When appropriate, use bullet points or numbered lists for better readability.\n\nRetrieved Context:\n{c

In [ ]:
from langchain_classic.chains import create_retrieval_chain
retriever = vectorestore.as_retriever(
    search_kwargs={"k": 3}
)

rag_chain = create_retrieval_chain(
    retriever,
    document_chain
)

rag_chain

In [45]:
response = rag_chain.invoke({"input":"Full form of ML"})
response['answer']

'Based on the provided context, the full form of **ML** is **Machine Learning**.'

## CREATING RAG CHAIN Alternative using LECL(LNGCHAIN EXPRESSION LANGUAGE)

In [47]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableParallel

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

custom_prompt = ChatPromptTemplate.from_template(
"""
You are a helpful AI assistant.

Use the following context to answer the user's question.

Rules:
- Answer only from the provided context.
- If the answer is not available in the context, simply say:
  "I don't know based on the provided context."
- Do not make up information.
- Support your answer with relevant details from the context whenever possible.

Context:
{context}

Question:
{input}

Answer:
"""
)

custom_prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\nYou are a helpful AI assistant.\n\nUse the following context to answer the user\'s question.\n\nRules:\n- Answer only from the provided context.\n- If the answer is not available in the context, simply say:\n  "I don\'t know based on the provided context."\n- Do not make up information.\n- Support your answer with relevant details from the context whenever possible.\n\nContext:\n{context}\n\nQuestion:\n{input}\n\nAnswer:\n'), additional_kwargs={})])

In [49]:
def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

In [60]:
rag_chain_lcel = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough(),
    }
    | custom_prompt
    | llm
    | StrOutputParser()
)

In [61]:
response = rag_chain_lcel.invoke("What is ML?")
print(response)

Based on the provided context, Machine Learning (ML) is a subset of Artificial Intelligence that enables computers to learn patterns from data without being explicitly programmed. 

Depending on the type of training data available, ML algorithms can be categorized as supervised, unsupervised, or reinforcement learning. It is used in applications such as recommendation systems, spam detection, fraud detection, predictive analytics, and image classification.
